In [ ]:
import numpy as np
import pandas as pd
import pandas.api.types

from typing import Optional


class ParticipantVisibleError(Exception):
    # If you want an error message to be shown to participants, you must raise the error as a ParticipantVisibleError
    # All other errors will only be shown to the competition host. This helps prevent unintentional leakage of solution data.
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Top-10 balanced accuracy for a multiclass classification task where the
    submission contains two probability distributions per row: one over the
    primary vocabulary (``vocabulary.csv``) and one over the secondary "moses"
    vocabulary (``moses-vocabulary.csv``). Only the primary distribution is
    scored; the secondary columns are carried for downstream use and dropped
    here.

    Solution format:   id, label, [Usage]
        - ``label`` is the word in the primary vocabulary.
    Submission format: id, <primary vocab probs...>, moses_<word> probs...
        - The primary-vocab columns appear first, in the same order as
          ``vocabulary.csv`` / the example submission.
        - The secondary distribution uses ``moses_`` as a column-name prefix
          (e.g. ``moses_am``, ``moses_are``, ...) and is excluded from scoring.

    Examples
    --------
    >>> import pandas as pd
    >>> row_id_column_name = "id"
    >>> y_true = pd.DataFrame({"id": [0, 1, 2, 3], "label": ['is', 'the', 'a', 'is']})
    >>> y_pred = pd.DataFrame({
    ...     "id":   [0, 1, 2, 3],
    ...     "is":   [0.7, 0.1, 0.2, 0.6],
    ...     "the":  [0.2, 0.2, 0.7, 0.3],
    ...     "a":    [0.1, 0.7, 0.1, 0.1],
    ... })
    >>> score(y_true.copy(), y_pred.copy(), row_id_column_name="test")
    0.06
    """

    # Pre-submission checks start
    # Check if number of columns are 51 or 101 (id,50(primary_vocab),50(moses_vocab))
    if row_id_column_name != "test":
        if submission.shape[1] != 51 and submission.shape[1] != 101:
            raise ParticipantVisibleError(
                f'The number of columns in the submission must be 51 (id + 50 primary_vocab) or 101 (id + 50 primay + 50 moses_vocab).')

    # Check if the number of samples in solution and submission match
    if solution.shape[0] != submission.shape[0]:
        raise ParticipantVisibleError(
            f'The number of samples in the solution and submission do not match. Please verify your submission.')

    # Check if the first column is stricly 'id'
    if submission.columns[0].lower() != 'id':
        raise ParticipantVisibleError(f'The first column of the submission must be "id".')

    # Check if all columns in the submission are numeric type
    for col in submission.columns:
        if not pd.api.types.is_numeric_dtype(submission[col]):
            raise ParticipantVisibleError(f'Submission column {col} must be a number')

    # Except 'id' column, all values must be between 0 and 1
    if not ((submission.iloc[:,1:] >=0) & (submission.iloc[:,1:] <=1)).all().all():
        raise ParticipantVisibleError('Some values do not seem to be between 0 and 1. Are you sure your probabilities are valid?')
    # Pre-submission checks over


    # Main classes that are used to calculate the balanced accuracy
    target_classes = ['is','the','a','to','it','i','not','was','we','be','he','that','have','this','they','of',
                      'there','and','are','in','but','will','so','all','my','for','she','were','any','really',
                      'at','out','our','am','its','had','him','an','very','has','do','can','time','think','good',
                      'always','new','people','as','on']

    # --- Keep ONLY the primary-vocabulary probability columns --------------------
    # First drop the secondary 'moses' columns, then drop the id / row-id column.
    # What remains are the 50 primary-vocab columns in vocabulary.csv order
    # (== target_classes order), so column position i corresponds to class code i.
    # FIX: previously the id column stayed in the array during argsort below, which
    # shifted every class index by one so the true label never lined up.
    submission = submission[[col for col in submission.columns if 'moses' not in col]]
    submission = submission.drop(columns=submission.columns[0])

    # Skip all rows where label is not from the above list
    # Convert labels to lowercase so that match target_labels
    solution['label'] = solution['label'].str.lower()
    filter_labels = solution['label'].isin(target_classes)
    solution_filtered = solution[filter_labels].copy()
    submission_filtered = submission[filter_labels]

    # Convert to numpy
    submission_np = submission_filtered.to_numpy()

    # Extract top-10 predictions for all samples: the 10 HIGHEST-probability classes.
    # FIX: previously np.argsort(submission_np)[:, :10] took the 10 LOWEST-probability
    # classes (ascending sort). Negating sorts descending -> the 10 largest.
    submission_np = np.argsort(-submission_np, axis=1)[:, :10]

    # Encode labels
    solution_filtered['label_encoded'] = (pd.Categorical(solution_filtered['label'], categories=target_classes, ordered=True)).codes

    # Check if the true label exist in top-10 predictions
    labels_encoded = solution_filtered['label_encoded'].to_numpy()
    correct = (submission_np == labels_encoded[:, None]).any(axis=1)

    # Calculate true number of instances per class and correct predictions per class
    count_per_class = np.bincount(labels_encoded, minlength=len(target_classes))
    correct_per_class = np.bincount(labels_encoded, weights=correct.astype(int), minlength=len(target_classes))

    # Remove classes with no samples in the solution as they don't contribute anything
    mask = count_per_class != 0
    count_per_class = count_per_class[mask]
    correct_per_class = correct_per_class[mask]

    # Per class accuracy
    accuracy = correct_per_class/ count_per_class

    # Total accuracy. We always divide by total number of target classes
    total_accuracy = np.sum(accuracy)/len(target_classes)

    return float(total_accuracy)
